# Build Baseline Process Dataset
Collect process snapshots from the current host (Windows/Linux compatible) to build a normal-behavior baseline for NGAV anomaly training.

In [11]:
from pathlib import Path
import hashlib
import platform
import psutil
import pandas as pd

In [12]:
def _safe(callable_obj, default=None):
    try:
        return callable_obj()
    except Exception:
        return default

def _sha256_file(path):
    if not path:
        return None
    p = Path(path)
    if not p.exists() or not p.is_file():
        return None
    try:
        h = hashlib.sha256()
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

def _extract_process_record(proc):
    with proc.oneshot():
        pid = proc.pid
        ppid = _safe(proc.ppid)
        name = _safe(proc.name, "") or ""
        exe = _safe(proc.exe, "") or ""
        username = _safe(proc.username, "") or ""
        cmdline = _safe(proc.cmdline, []) or []
        status = _safe(proc.status, "unknown") or "unknown"
        create_time = _safe(proc.create_time, 0.0) or 0.0
        cpu_percent = _safe(lambda: proc.cpu_percent(interval=0.0), 0.0) or 0.0
        memory_info = _safe(proc.memory_info)
        memory_rss = getattr(memory_info, 'rss', 0) if memory_info else 0
        num_threads = _safe(proc.num_threads, 0) or 0
        num_fds = _safe(proc.num_fds, -1)
        if num_fds is None:
            num_fds = -1

    lower_exe = exe.lower()
    is_system_path = int('windows' in lower_exe and ('\\windows\\system32' in lower_exe or '\\windows\\syswow64' in lower_exe))
    if platform.system().lower() != 'windows':
        is_system_path = int(lower_exe.startswith('/usr/') or lower_exe.startswith('/bin/') or lower_exe.startswith('/sbin/'))

    is_temp_path = int('\\temp\\' in lower_exe or '/tmp/' in lower_exe)

    return {
        'pid': pid,
        'ppid': ppid,
        'name': name,
        'exe': exe,
        'username': username,
        'status': status,
        'create_time': create_time,
        'cpu_percent': cpu_percent,
        'memory_rss': memory_rss,
        'num_threads': num_threads,
        'num_fds': num_fds,
        'cmdline_len': len(' '.join(cmdline)),
        'is_system_path': is_system_path,
        'is_temp_path': is_temp_path,
        'platform': platform.system().lower(),
        'exe_sha256': _sha256_file(exe)
    }

In [13]:
rows = []
for proc in psutil.process_iter():
    record = _safe(lambda: _extract_process_record(proc))
    if record is not None:
        rows.append(record)

df = pd.DataFrame(rows)
df.head()

,pid,ppid,name,exe,username,status,create_time,cpu_percent,memory_rss,num_threads,num_fds,cmdline_len,is_system_path,is_temp_path,platform,exe_sha256
0,1,0,systemd,/usr/lib/systemd/systemd,root,sleeping,1.780307e+09,0.0,15929344,1,-1,73,1,0,linux,594f5de1a2b5eeb3650bc7a77a7e3137abed3a621c7744...
1,2,0,kthreadd,,root,sleeping,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
2,3,2,pool_workqueue_release,,root,sleeping,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
3,4,2,kworker/R-rcu_gp,,root,idle,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
4,5,2,kworker/R-sync_wq,,root,idle,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN


In [14]:
output_path = Path('normal_processes.csv')
df.to_csv(output_path, index=False)
print(f'Saved {len(df)} process records to {output_path.resolve()}')

Saved 309 process records to /home/phuong/btl_ngav/notebooks/normal_processes.csv
